In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_json, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType
from pyspark.sql import functions as F
import os

In [2]:
try:
    spark.stop()
except:
    pass

In [3]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Patient")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 20:27:54 WARN Utils: Your hostname, Hariprasads-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.11.116.189 instead (on interface en0)
26/03/09 20:27:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 20:27:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
bronze_path = "../../data_lake/bronze/batch_data/resource_type=Patient/"
silver_base_path = "../../data_lake/silver/silver_Patient/"

In [4]:
#schema for patient resource
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

communication_schema = ArrayType(StructType([
    StructField("language", StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ]), True),
    StructField("preferred", StringType(), True)
    
]))

#Marital schema

marital_schema = (StructType([
    StructField("coding", ArrayType(general_coding_schema), True),
    StructField("text", StringType(), True)
]))

#telecom schema

telecom_schema = ArrayType(StructType([
    StructField("system", StringType(), True),
    StructField("value", StringType(), True),
    StructField("use", StringType(), True)
]))

# more will be added later


In [5]:
#identifier schema
identifier_coding_schema = StructType([
    StructField("coding", ArrayType(general_coding_schema), True),
    StructField("text",   StringType(), True),
])


patient_identifier_type = ArrayType(
    StructType([
        StructField("type",   identifier_coding_schema, True),  
        StructField("system", StringType(), True),  
        StructField("value",  StringType(), True),  
    ])
)

def extract_patient_identifier(type_code:str):
    return F.filter(lambda x: x['type']['coding'][0]['code'] == type_code, col("identifier"))[0]['value']

In [6]:
#name schema
patient_name_schema = ArrayType(
    StructType([
        StructField("use", StringType(), True),  
        StructField("family", StringType(), True),
        StructField("given", ArrayType(StringType()), True),
        StructField("prefix", ArrayType(StringType()), True),
        StructField("suffix", ArrayType(StringType()), True)
    ])
)

def extract_patient_name(name_part:str):
    if name_part == "family":
        return col("patient_name")[0]['family']
    elif name_part == "first_name":
        return col("patient_name")[0]['given'][0]
    elif name_part == "middle_name":
        return col("patient_name")[0]['given'][1] if F.size(col("patient_name")[0]['given']) > 1 else None
    elif name_part == "prefix":
        return col("patient_name")[0]['prefix'][0]
    elif name_part == "suffix":
        return col("patient_name")[0]['suffix'][0]
    else:
        return None

In [7]:
#address schema
address_extension_schema = ArrayType(StructType([
    # StructField("url", StringType(), True), i dont need it now 
    StructField("extension", ArrayType(
        StructType([
            StructField("url",StringType(),True),
            StructField("valueDecimal",DoubleType(),True)
        ])
    ), True)
])
)
address_schema = ArrayType(
    StructType([
        StructField("extension",address_extension_schema, True),
        StructField("line", ArrayType(StringType()), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True),
        StructField("country", StringType(), True)
    ])
)

In [ ]:
#extension schema which contains few details of patients - race, ethnicity, etc
extension_schema = ArrayType(StructType([
    StructField("url", StringType(), True),
    StructField("extension", ArrayType(
        StructType([
            StructField("url",StringType(),True),
            StructField("valueString",StringType(),True)
        ])
    ), True),
    StructField("valueString", StringType(), True),
    StructField("valueCode", StringType(), True),
    StructField("valueAddress", address_schema, True),  # reusing the address schema because city, state and country keys are common
]))

In [ ]:
df_patient = spark.read.format("parquet").load(bronze_path).withColumn("silver_timestamp", current_timestamp())
df_patient.printSchema()

In [ ]:
df_intermediate_1 = df_patient.select(
    col("fullUrl").alias("patient_fhir_id"),
    F.conv(F.substring(F.md5("fullUrl"), 1, 15), 16, 10).cast("bigint").alias("patient_key"),
    col("resource.id").alias("patient_id"),
    col("resource.extension").alias("extension"),
    from_json(col("resource.identifier"), patient_identifier_type).alias("identifier"),
    from_json(col("resource.name"), patient_name_schema).alias("patient_name"),
    from_json(col("resource.telecom"),telecom_schema)[0]["value"].alias("phone_number"),
    col("resource.gender").alias("gender"),
    col("resource.birthDate").alias("birthDate"),
    col("resource.deceasedDateTime").alias("deceasedDateTime"),
    from_json(col("resource.address"), address_schema).alias("patient_address"),
    col("resource.maritalStatus").alias("maritalStatus"),
    col("resource.multipleBirthBoolean").cast("boolean").alias("is_multiple_birth"),
    col("resource.communication").alias("communication")
).drop("resource","ingestion_timestamp")

In [ ]:
df_int_2 = (df_intermediate_1
            .select(
                col("patient_fhir_id"),
                col("patient_key"),
                col("patient_id"),
                extract_patient_identifier("MR").alias("medical_record_number"),
                extract_patient_identifier("SS").alias("social_security_number"),
                extract_patient_identifier("DL").alias("drivers_license_number"),
                extract_patient_identifier("PPN").alias("passport_number"),
                extract_patient_name("first_name").alias("first_name"),
                extract_patient_name("middle_name").alias("middle_name"),
                extract_patient_name("family").alias("family_name"),
                extract_patient_name("prefix").alias("prefix"),
                # extract_patient_name("suffix").alias("suffix"),   if needed we can include later
                col("gender"),
                col("birthDate").cast("date").alias("birth_date"),
                col("deceasedDateTime").cast("timestamp").alias("deceased_date_time"),
                F.array_join(col("patient_address")[0], ", ").alias("address_line"),
                col("patient_address")[0]['city'].alias("city"),
                col("patient_address")[0]['state'].alias("state"),
                col("patient_address")[0]['postalCode'].alias("postal_code"),
                col("patient_address")[0]['country'].alias("country"),
                col("patient_address")[0]['extension'][0]['extension'][0]['valueDecimal'].alias("latitude"),
                col("patient_address")[0]['extension'][0]['extension'][1]['valueDecimal'].alias("longitude"),
                col("phone_number"),
                explode_outer(col("patient_telecom")).alias("patient_telecom_exploded"),
            )
)

In [ ]:
df_intermediate_2 = (
    df_intermediate_1.withColumn(
        "communication", from_json(col("communication"), communication_schema)
    )
    .withColumn(
        "patient_language_code",
        col("communication")[0]["language"]["coding"][0]["code"],
    )
    .withColumn(
        "patient_language", col("communication")[0]["language"]["coding"][0]["display"]
    )
    .withColumn("maritalStatus_parsed", from_json(col("maritalStatus"), marital_schema))
    .withColumn("marital_status_code", col("maritalStatus_parsed")["coding"][0]["code"])
    .withColumn(
        "marital_status_display", col("maritalStatus_parsed")["coding"][0]["display"]
    )
).drop("maritalStatus_parsed", "maritalStatus")


In [ ]:
df_intermediate_2.write.mode("overwrite").option("overwriteSchema", "true").parquet(silver_base_path)

In [ ]:
try:
    spark.stop()
except:
    pass